# 2026-06-27 — Genome summary table: unique strains (representatives)

A condensed version of the Table 1 genome summary from `0626_genome_comparison`. Our five sequenced
isolates collapse into **three genetically distinct strains** (by whole-genome ANI):

| Strain | Isolates (≈100% ANI clones) | Species |
|---|---|---|
| 1 | **8-5-2**, 10-10-2 | *Pseudomonas_E* sp013179515 |
| 2 | **9-2-1**, 9-6-2 | *Pseudomonas_E* sp013179515 |
| 3 | **8-10-2** | *Pseudomonas_E capeferrum* |

This notebook picks **one representative per strain** (best-quality genome) and tabulates only those
three, so each unique strain is shown once. Run from `code/`. Output → `output/0627_unique_strain_table/`.

In [1]:
import glob
from pathlib import Path
import pandas as pd

OUT = Path("output/0627_unique_strain_table"); OUT.mkdir(parents=True, exist_ok=True)

STRAIN_DIR = {   # all five isolates (we choose representatives below)
    "8-5-2":   "../data/bacteria_genome/8-5-2",
    "10-10-2": "../data/bacteria_genome/10-10-2",
    "9-2-1":   "../data/bacteria_genome/9-2-1",
    "9-6-2":   "../data/bacteria_genome/9-6-2/rerun_QB9Y48",
    "8-10-2":  "../data/bacteria_genome/8-10-2",
}

# the three ANI-defined strains (≈100% within each); first member is a placeholder, rep chosen by quality
CLUSTERS = {
    "Strain 1 (sp013179515)": ["8-5-2", "10-10-2"],
    "Strain 2 (sp013179515)": ["9-2-1", "9-6-2"],
    "Strain 3 (P. capeferrum)": ["8-10-2"],
}

SITE_META = {   # field metadata keyed by site group (first token of the isolate code)
    "8":  {"coords": "34\u00b06\'54\"N 117\u00b043\'28\"W", "elev": "410", "plant": "Chilopsis linearis"},
    "9":  {"coords": "34\u00b06\'54\"N 117\u00b043\'28\"W", "elev": "410", "plant": "Aesculus californica"},
    "10": {"coords": "34\u00b06\'52\"N 117\u00b043\'13\"W", "elev": "410", "plant": "Nicotiana quadrivalvis"},
}

def _one(d, pat):
    g = glob.glob(f"{d}/{pat}")
    assert g, f"no file matches {d}/{pat}"
    return g[0]

def parse_bakta_txt(path):
    d = {}
    for line in Path(path).read_text().splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            d[k.strip()] = v.strip()
    return d

def parse_kv_tsv(path):
    out = {}
    for line in Path(path).read_text().splitlines():
        p = line.rstrip("\n").split("\t")
        if len(p) >= 2:
            out[p[0]] = p[1]
    return out

def parse_header_tsv(path):
    rows = Path(path).read_text().splitlines()
    return dict(zip(rows[0].split("\t"), rows[1].split("\t")))

def parse_sourmash(path):
    rows = Path(path).read_text().splitlines()
    r = dict(zip(rows[0].split(","), rows[1].split(",")))
    return r["lineage"].split(";s__")[-1], float(r["query_ani_at_rank"])

def topology(path):
    t = Path(path).read_text()
    return "Circular" if "topology: circular" in t else ("Linear" if "topology:" in t else "\u2014")

print("isolates available:", list(STRAIN_DIR))

isolates available: ['8-5-2', '10-10-2', '9-2-1', '9-6-2', '8-10-2']


## Representative selection

For each clone pair we keep the higher-quality genome: **highest CheckM completeness, then lowest
contamination, then highest sequencing depth (coverage)**.

In [2]:
def quality(strain):
    chk = parse_header_tsv(_one(STRAIN_DIR[strain], "*_checkm-results.tsv"))
    ont = parse_kv_tsv(_one(STRAIN_DIR[strain], "*_ont-stats.tsv"))
    return (float(chk["Completeness"]), -float(chk["Contamination"]), float(ont["est_cov"]))

REPS, MEMBERS = {}, {}
for label, members in CLUSTERS.items():
    rep = max(members, key=quality)
    REPS[label] = rep
    MEMBERS[rep] = members
    detail = ", ".join(
        f"{m}{' *<- rep*' if m == rep else ''} "
        f"(compl {quality(m)[0]:.2f}, contam {-quality(m)[1]:.2f}, cov {quality(m)[2]:.0f}x)"
        for m in members)
    print(f"{label}: {detail}")

REP_ORDER = list(REPS.values())          # e.g. ['8-5-2', '9-2-1', '8-10-2']
print("\nrepresentatives ->", REP_ORDER)

Strain 1 (sp013179515): 8-5-2 *<- rep* (compl 99.74, contam 0.87, cov 95x), 10-10-2 (compl 99.74, contam 0.87, cov 84x)
Strain 2 (sp013179515): 9-2-1 *<- rep* (compl 99.74, contam 1.94, cov 94x), 9-6-2 (compl 99.74, contam 1.94, cov 67x)
Strain 3 (P. capeferrum): 8-10-2 *<- rep* (compl 100.00, contam 0.20, cov 96x)

representatives -> ['8-5-2', '9-2-1', '8-10-2']


## Table — one representative per unique strain

Same metrics as the `0626` Table 1, but only the three representative genomes. A **“Represents”**
row lists the ≈100%-ANI clones each representative stands in for.

In [3]:
def collect(strain):
    d = STRAIN_DIR[strain]
    bak = parse_bakta_txt(_one(d, "annotation/*_reference.txt"))
    ont = parse_kv_tsv(_one(d, "*_ont-stats.tsv"))
    chk = parse_header_tsv(_one(d, "*_checkm-results.tsv"))
    sp, ani = parse_sourmash(_one(d, "*_sourmash-genome-species.csv"))
    sample_id = Path(_one(d, "*_ont-stats.tsv")).name.replace("_ont-stats.tsv", "")
    total_genes = sum(int(bak[k]) for k in ["CDSs", "tRNAs", "tmRNAs", "rRNAs", "ncRNAs"])
    meta = SITE_META[strain.split("-")[0]]
    return {
        "Isolate (internal name)":     strain,
        "Sequencing sample ID":        sample_id,
        "Represents (\u2248100%-ANI clones)": ", ".join(MEMBERS[strain]),
        "Genome accession (GenBank)":  "pending",
        "Raw read accession (SRA)":    "pending",
        "Isolation source":            "California Botanic Garden soil",
        "Coordinates":                 meta["coords"],
        "Elevation (m)":               meta["elev"],
        "Adjacent plant":              meta["plant"],
        "Sequencing platform":         "Oxford Nanopore (long-read)",
        "Genome size (bp)":            f"{int(bak['Length']):,}",
        "Contigs (n)":                 bak["Count"],
        "Topology":                    topology(_one(d, "*_prepolished-assembly-info.yaml")),
        "N50 (bp)":                    f"{int(bak['N50']):,}",
        "GC content (%)":              bak["GC"],
        "Sequencing depth (\u00d7)":       f"{float(ont['est_cov']):.0f}",
        "Total reads":                 f"{int(ont['total_num_reads']):,}",
        "Read N50 (bp)":               f"{int(float(ont['rN50'])):,}",
        "Median read length (bp)":     f"{int(float(ont['median_read'])):,}",
        "Median read quality (Phred)": ont["median_phred"],
        "Completeness % (CheckM)":     chk["Completeness"],
        "Contamination % (CheckM)":    chk["Contamination"],
        "Closest species (GTDB)":      sp,
        "ANI to closest (%)":          f"{ani * 100:.2f}",
        "Total genes":                 f"{total_genes:,}",
        "CDS (protein-coding genes)":  f"{int(bak['CDSs']):,}",
        "tRNA":                        bak["tRNAs"],
        "rRNA":                        bak["rRNAs"],
        "ncRNA":                       bak["ncRNAs"],
        "Pseudogenes":                 bak["pseudogenes"],
        "Hypothetical proteins":       bak["hypotheticals"],
        "Coding density (%)":          bak["coding density"],
    }

records = {rep: collect(rep) for rep in REP_ORDER}
table = pd.DataFrame(records).reindex(list(records[REP_ORDER[0]].keys()))
table.index.name = "Metric"
table.to_csv(OUT / "strain_summary_representatives.csv")
table

,8-5-2,9-2-1,8-10-2
Metric,,,
Isolate (internal name),8-5-2,9-2-1,8-10-2
Sequencing sample ID,B9TF7H_1_Sample_1_8A,XH254M_1_sample_1,CZJZZ4_1_rerun_Sample_2_8B_2
Represents (≈100%-ANI clones),"8-5-2, 10-10-2","9-2-1, 9-6-2",8-10-2
Genome accession (GenBank),pending,pending,pending
Raw read accession (SRA),pending,pending,pending
Isolation source,California Botanic Garden soil,California Botanic Garden soil,California Botanic Garden soil
Coordinates,"34°6'54""N 117°43'28""W","34°6'54""N 117°43'28""W","34°6'54""N 117°43'28""W"
Elevation (m),410,410,410
Adjacent plant,Chilopsis linearis,Aesculus californica,Chilopsis linearis


## Summary

Three unique strains shown, one representative genome each (clone members collapsed by ≈100% ANI).
Output: `output/0627_unique_strain_table/strain_summary_representatives.csv`.

---
*Log: 2026-06-27 — created. Condensed genome summary showing only the 3 ANI-distinct strains (reps: 8-5-2 for {8-5-2, 10-10-2}; 9-2-1 for {9-2-1, 9-6-2}; 8-10-2 for P. capeferrum). Representative = best genome per clone pair (CheckM completeness → contamination → coverage). Derived from the 5-isolate Table 1 in `0626_genome_comparison`.*